## Test profiling package

- Aim: find a good profiling package to use
- I want:
1. Benchmark code
2. Show size of data
3. Show each line and cost per
line

## Function to test

In [ ]:
%run "/home/user/python-pannguyen/lab_file/function_to_test.ipynb"

In [ ]:
%prun heavy_function(6000)

### cProfile

In [ ]:
import cProfile
import pstats

with cProfile.Profile() as pr:
    heavy_function(6000)

stats = pstats.Stats(pr)
stats.sort_stats(pstats.SortKey.TIME)
stats.print_stats(10)

In [ ]:
import cProfile, pstats

class _ProfileFunc:
    def __init__(self, func, sort_stats_by):
        self.func =  func
        self.profile_runs = []
        self.sort_stats_by = sort_stats_by

    def __call__(self, *args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()  # this is the profiling section
        retval = self.func(*args, **kwargs)
        pr.disable()

        self.profile_runs.append(pr)
        ps = pstats.Stats(*self.profile_runs).sort_stats(self.sort_stats_by)
        return retval, ps

def cumulative_profiler(amount_of_times, sort_stats_by='time'):
    def real_decorator(function):
        def wrapper(*args, **kwargs):
            nonlocal function, amount_of_times, sort_stats_by  # for python 2.x remove this row

            profiled_func = _ProfileFunc(function, sort_stats_by)
            for i in range(amount_of_times):
                retval, ps = profiled_func(*args, **kwargs)
            ps.print_stats()
            return retval  # returns the results of the function
        return wrapper

    if callable(amount_of_times):  # incase you don't want to specify the amount of times
        func = amount_of_times  # amount_of_times is the function in here
        amount_of_times = 5  # the default amount
        return real_decorator(func)
    return real_decorator

In [ ]:
import time

@cumulative_profiler
def baz():
    time.sleep(1)
    time.sleep(2)
    return 1

baz()

In [32]:
import cProfile
import pstats
import io
from functools import wraps

def profile(func=None, output_file=None):
    def decorator(f):
        @wraps(f)
        def wrapper(*args, **kwargs):
            # Create and start profiler
            pr = cProfile.Profile()
            pr.enable()

            # Call the original function
            result = f(*args, **kwargs)

            # Stop profiling
            pr.disable()

            # Print formatted results to console
            s = io.StringIO()
            ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
            ps.print_stats(20)
            print(s.getvalue())

            # Save to file if requested
            if output_file:
                ps.dump_stats(output_file)
                print(f"Profile data saved to {output_file}")

            return result
        return wrapper

    # Handle both @profile and @profile(output_file='stats.prof') syntax
    if func is None:
        return decorator
    return decorator(func)


In [33]:
@profile(output_file='heavy_function.lprof')
def run():
    heavy_function(6_000)

run()

         2833435 function calls (2833410 primitive calls) in 2.973 seconds

   Ordered by: cumulative time
   List reduced from 28 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.033    0.033    2.973    2.973 /tmp/ipykernel_20912/3231244301.py:1(run)
        1    0.010    0.010    2.940    2.940 /tmp/ipykernel_20912/2620111589.py:7(heavy_function)
        1    0.517    0.517    2.507    2.507 /tmp/ipykernel_20912/2620111589.py:24(<listcomp>)
   300000    0.311    0.000    1.913    0.000 /usr/lib/python3.11/random.py:358(randint)
   300000    0.812    0.000    1.602    0.000 /usr/lib/python3.11/random.py:284(randrange)
   300000    0.455    0.000    0.628    0.000 /usr/lib/python3.11/random.py:235(_randbelow_with_getrandbits)
        1    0.194    0.194    0.194    0.194 {method 'sort' of 'list' objects}
        3    0.190    0.063    0.190    0.063 {built-in method time.sleep}
   900000    0.161    0.000    0.161   

### PyInstrument

In [ ]:
# !pip install pyinstrument

In [ ]:
from pyinstrument import Profiler
with Profiler(interval=0.1) as profiler:
    heavy_function(n=6_000)

In [ ]:
profiler.print()

In [ ]:
profiler.open_in_browser()

### memory_profiler

In [ ]:
!pip install memory_profiler


In [ ]:
from memory_profiler import profile

@profile
def test():
    heavy_function(6_000)

test()

In [ ]:
%load_ext memory_profiler


In [ ]:
%memit heavy_function(6_000)

### pympler

In [ ]:
!pip install pympler

### auto_profiler

In [ ]:
!pip install auto_profiler


In [ ]:
from auto_profiler import Profiler
with Profiler(depth=4):
    heavy_function(6000)

### pprofile

In [ ]:
# !pip install pprofile

In [ ]:
import pprofile
prof = pprofile.StatisticalProfile()
with prof():
    heavy_function(6000)
    prof.print_stats()

### Scalene

In [ ]:
!pip install -U scalene


In [ ]:
from scalene import profile

@profile
def test():
    heavy_function(6000)

test()

### line_profiler

In [ ]:
!pip install line_profiler

In [ ]:
%load_ext line_profiler


In [ ]:
%lprun heavy_function(6000)